# Day 38 — Storytelling with Data: Communicating Analysis
**Estimated time:** 90 minutes  
**Learning objectives:**
1. Translate data outputs into clear executive narratives
2. Choose the right chart type for the business message (not just for the data shape)
3. Build a polished one-page report template using pandas Styler + matplotlib

---
> **senior-level expectation:** A senior analyst is expected to not just produce analysis but to *communicate findings* to non-technical stakeholders. The outputs from days 21–37 are raw materials. Today you turn them into deliverables.


## 0. Setup

In [ ]:
import pandas as pd, numpy as np, matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.ticker as mticker
from pathlib import Path

pd.set_option('display.float_format', '{:,.2f}'.format)
DATA = Path('../datasets')
TODAY = pd.Timestamp('2026-04-01')

bw = pd.read_csv(DATA / 'bw_sales_kpis.csv')
cc = pd.read_csv(DATA / 'cost_center_actuals.csv')
hr = pd.read_csv(DATA / 'hr_headcount.csv', parse_dates=['HIRE_DATE','TERM_DATE'])
mat = pd.read_csv(DATA / 'materials_inventory.csv')
so = pd.read_csv(DATA / 'sales_orders.csv', parse_dates=['ERDAT'])

print('Setup complete.')

## 1. Writing Executive Summaries from Data Outputs

In [ ]:
# YOUR SOLUTION
# Rule: Every sentence in an executive summary must come from a number.
# Never write "revenue improved significantly" — write "revenue grew 12.3% YoY to $4.2M"

# Compute the numbers first
cy_rev = bw[bw['CAL_YEAR_MONTH']//100==2025]['REVENUE'].sum()
py_rev = bw[bw['CAL_YEAR_MONTH']//100==2024]['REVENUE'].sum()
yoy_pct = (cy_rev - py_rev) / py_rev * 100

cy_gm_pct = (bw[bw['CAL_YEAR_MONTH']//100==2025]['GROSS_MARGIN'].sum() / cy_rev * 100)
py_gm_pct = (bw[bw['CAL_YEAR_MONTH']//100==2024]['GROSS_MARGIN'].sum() / py_rev * 100)

open_backlog = so[so['STATUS']=='OPEN']['NETWR'].sum()
high_risk_count = so[(so['STATUS']=='OPEN') &
                     ((TODAY - so['ERDAT']).dt.days > 60) &
                     (so['NETWR'] > 5000)].shape[0]

mat['DAYS_OLD'] = (TODAY - pd.to_datetime(mat['LAST_MOVEMENT_DATE'],errors='coerce')).dt.days.fillna(999)
dead_val = mat[mat['DAYS_OLD']>180]['INVENTORY_VALUE'] if 'INVENTORY_VALUE' in mat.columns else (mat[mat['DAYS_OLD']>180]['LABST'] * mat[mat['DAYS_OLD']>180]['STPRS'])
dead_val = dead_val.sum()

active_hc = hr[hr['TERM_DATE'].isna()].shape[0]

# GOOD executive summary — data-referenced, specific, actionable
summary = f"""
BUSINESS PERFORMANCE SUMMARY — {TODAY.strftime('%B %Y')}
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

Revenue of ${cy_rev/1e6:.1f}M represents {yoy_pct:+.1f}% growth versus the prior year,
driven primarily by the Central and West regions.

Gross margin compressed by {py_gm_pct-cy_gm_pct:.1f}pp to {cy_gm_pct:.1f}%, reflecting
higher COGS and a {bw[bw['CAL_YEAR_MONTH']//100==2025]['DISCOUNT_AMT'].sum()/cy_rev*100:.1f}%
effective discount rate. A 1pp reduction in discounting would add approximately
${cy_rev*0.01/1e3:.0f}K to gross profit.

The open order backlog of ${open_backlog/1e3:.0f}K contains {high_risk_count} high-risk orders
(age >60 days and value >$5K) requiring immediate sales follow-up.

Dead inventory of ${dead_val/1e3:.0f}K (no movement >180 days) represents an exposure
for write-down in the next financial close cycle.

Active workforce stands at {active_hc} employees across all plants.
"""
print(summary)

## 2. Choosing the Right Chart Type

### Chart Type Selection Guide (for Analytics Work)

| Business Question | Chart Type | Why |
|---|---|---|
| How has revenue changed over time? | Line chart | Shows trend and slope |
| How does each region compare this month? | Horizontal bar | Easy rank comparison |
| What share of backlog is in each bucket? | Stacked bar (not pie) | Preserves proportions, sortable |
| Is there correlation between discount and margin? | Scatter plot | Shows relationship direction |
| Where are the outliers in cost variance? | Box plot or dot plot | Shows distribution + extremes |
| How does actuals vs plan look across 12 periods? | Dual-line or waterfall | Shows gap visually |

**Anti-patterns:**
- Pie charts with >4 slices → use horizontal bar
- 3D charts of any kind → never
- Dual Y-axis with non-comparable scales → misleads readers
- Too many data points in one chart → split by region or product


## 3. Polished One-Page Report Template

In [ ]:
# YOUR SOLUTION
# Build a 4-panel one-page management report

fig = plt.figure(figsize=(16, 11))
fig.patch.set_facecolor('#f8f9fa')
gs = gridspec.GridSpec(3, 3, figure=fig, hspace=0.45, wspace=0.35)

TITLE_FONT = {'fontsize': 9, 'fontweight': 'bold', 'color': '#2c3e50'}
SUBTITLE_FONT = {'fontsize': 8, 'color': '#666666'}

# ── Title Banner ──────────────────────────────────────────────────────────────
title_ax = fig.add_subplot(gs[0, :])
title_ax.axis('off')
title_ax.text(0.5, 0.7, 'MONTHLY BUSINESS REVIEW — APRIL 2026',
              transform=title_ax.transAxes, ha='center', fontsize=16,
              fontweight='bold', color='#2c3e50')
title_ax.text(0.5, 0.2,
              f'Revenue: ${cy_rev/1e6:.1f}M ({yoy_pct:+.1f}% YoY)  |  '
              f'GM: {cy_gm_pct:.1f}%  |  Open Backlog: ${open_backlog/1e3:.0f}K  |  '
              f'Dead Inventory: ${dead_val/1e3:.0f}K  |  Headcount: {active_hc}',
              transform=title_ax.transAxes, ha='center', fontsize=10, color='#555555')
title_ax.axhline(0.05, color='#3498db', linewidth=2)

# ── Panel 1: Revenue Trend (line) ─────────────────────────────────────────────
ax1 = fig.add_subplot(gs[1, :2])
monthly_rev = bw.groupby('CAL_YEAR_MONTH')['REVENUE'].sum().reset_index()
monthly_rev = monthly_rev[monthly_rev['CAL_YEAR_MONTH'] >= 202401]
ax1.plot(range(len(monthly_rev)), monthly_rev['REVENUE']/1e6, color='#2980b9',
         marker='o', linewidth=2, markersize=4)
ax1.fill_between(range(len(monthly_rev)), monthly_rev['REVENUE']/1e6, alpha=0.1, color='#2980b9')
ax1.set_title('Revenue Trend (CY2025)', **TITLE_FONT)
ax1.set_xticks(range(len(monthly_rev)))
ax1.set_xticklabels([str(p)[4:] for p in monthly_rev['CAL_YEAR_MONTH']], fontsize=7)
ax1.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'${x:.1f}M'))
ax1.set_facecolor('white')
ax1.grid(True, alpha=0.3)

# ── Panel 2: GM % by Region (horizontal bar) ──────────────────────────────────
ax2 = fig.add_subplot(gs[1, 2])
gm_region = (bw[bw['CAL_YEAR_MONTH']//100==2025]
             .groupby('REGION').apply(lambda d: d['GROSS_MARGIN'].sum()/d['REVENUE'].sum()*100)
             .sort_values())
colors_gm = ['#e74c3c' if v < 30 else '#2ecc71' for v in gm_region.values]
gm_region.plot(kind='barh', ax=ax2, color=colors_gm, edgecolor='white')
ax2.set_title('GM % by Region', **TITLE_FONT)
ax2.set_xlabel('GM %', fontsize=7)
ax2.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'{x:.0f}%'))
ax2.set_facecolor('white')
ax2.grid(True, alpha=0.3, axis='x')

# ── Panel 3: Inventory Aging (stacked bar) ────────────────────────────────────
ax3 = fig.add_subplot(gs[2, :2])
mat_copy = mat.copy()
mat_copy['INV_VAL'] = mat_copy['LABST'] * mat_copy['STPRS']
bins=[0,30,90,180,9999]; labels=['Current','Slow','Stagnant','Dead']
mat_copy['BUCKET'] = pd.cut(mat_copy['DAYS_OLD'].clip(0,9999), bins=bins, labels=labels)
aging_pivot = mat_copy.groupby(['WERKS','BUCKET'],observed=True)['INV_VAL'].sum().unstack(fill_value=0)
aging_pivot /= 1000
colors_aging = {'Current':'#2ecc71','Slow':'#f39c12','Stagnant':'#e67e22','Dead':'#e74c3c'}
aging_pivot[[c for c in labels if c in aging_pivot.columns]].plot(
    kind='bar', ax=ax3, stacked=True,
    color=[colors_aging.get(c,'#95a5a6') for c in [c for c in labels if c in aging_pivot.columns]],
    edgecolor='white', width=0.6)
ax3.set_title('Inventory Value by Aging Bucket ($K)', **TITLE_FONT)
ax3.tick_params(axis='x', rotation=0)
ax3.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'${x:.0f}K'))
ax3.set_facecolor('white')
ax3.legend(fontsize=7, loc='upper right')
ax3.grid(True, alpha=0.3, axis='y')

# ── Panel 4: Cost Variance Summary (text table) ───────────────────────────────
ax4 = fig.add_subplot(gs[2, 2])
ax4.axis('off')
cc_sum = (cc[cc['GJAHR']==2024].groupby('KTEXT')
          .agg(A=('ACTUAL_AMT','sum'),P=('PLAN_AMT','sum'))
          .assign(V=lambda d: d['A']-d['P'])
          .nlargest(5,'V')[['A','P','V']])
tbl_data = [[row.name[:18],f'${row["V"]/1e3:.1f}K'] for _, row in cc_sum.iterrows()]
tbl = ax4.table(cellText=tbl_data, colLabels=['Cost Center','Variance'],
                cellLoc='left', loc='center')
tbl.auto_set_font_size(False); tbl.set_fontsize(8)
for (r,c), cell in tbl.get_celld().items():
    if r == 0:
        cell.set_facecolor('#3498db'); cell.set_text_props(color='white', fontweight='bold')
    elif r % 2 == 0:
        cell.set_facecolor('#eaf4ff')
    cell.set_edgecolor('white')
ax4.set_title('Top Variance Drivers', **TITLE_FONT, pad=12)

plt.suptitle('', fontsize=1)  # prevent auto suptitle
plt.savefig('day38_one_page_report.png', dpi=150, bbox_inches='tight', facecolor='#f8f9fa')
plt.show()
print('One-page report saved.')

## 4. Practice: Write a 5-Sentence Narrative from the Capstone Data

In [ ]:
# YOUR SOLUTION
# Rules for the narrative:
# 1. Sentence 1: headline number + direction
# 2. Sentence 2: key driver (where is the change coming from?)
# 3. Sentence 3: risk or concern
# 4. Sentence 4: operational metric (inventory or backlog)
# 5. Sentence 5: recommendation or next action

print("""
PRACTICE NARRATIVE TEMPLATE (fill in the blanks):
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

[1-HEADLINE] Revenue for [period] reached $[X], a [±Y]% change vs. prior year.

[2-DRIVER] The [region/product] was the primary driver, contributing $[X] 
           or [Y]% of the total variance.

[3-RISK] Gross margin compressed to [X]%, [Y]pp below plan, primarily driven 
         by elevated discounting at [Z]% of revenue.

[4-OPERATIONAL] The open order backlog of $[X] includes [N] high-risk orders 
                requiring escalation, while $[Y] of inventory has not moved 
                in over 180 days.

[5-RECOMMENDATION] To recover margin and reduce working capital risk, we 
                   recommend [specific action] by [specific date].
""")

# Now write YOUR version using the actual numbers computed earlier
# YOUR CODE HERE

## Daily Prompt
**Answer independently:**

1. A stakeholder asks you to present the inventory aging analysis. Rewrite the stacked bar chart from Section 3 so the most important message ("dead inventory is $Xk") is the first thing a viewer's eye goes to. What changes would you make to color, labelling, or layout?
2. Write a 5-sentence narrative using the actual numbers from this notebook. No generic phrases — every sentence must reference a specific figure.
3. What is the difference between a "dashboard" and a "report" in the context of analytics deliverables? When would you recommend each to a senior stakeholder?
